## Pipeline for machine learning baselines

In [1]:
import os

# Check if it's in the correct directory
print("Current working directory:", os.getcwd())
path = os.path.abspath(os.path.join(os.getcwd(), '..', 'path.py'))
%run $path

Current working directory: c:\Users\vinic\Dropbox\baseline\baseline\notebooks


##### Configure notebook

In [2]:
# Import data
train_file = '../data/mico/train.csv'
val_file = '../data/mico/val.csv'
test_file = '../data/mico/test.csv'

# Fingerprints (ECFP4 = Morgan radius 2)
fp_size = 2048
radius = 2

# ML method: 'svm' or 'rf'
method = 'rf'
seed = 42
n_splits = 5
c_value = 0.0001

# Save trained models and metadata
model_out_dir = '../output/models_ml'
params_out_dir = '../output/params_ml'

##### Load data

In [3]:
from params import load_data

train_smiles, y_train = load_data(train_file)
val_smiles, y_val = load_data(val_file)
test_smiles, y_test = load_data(test_file)

print(f"Training data: {len(train_smiles)} samples")
print(f"Validation data: {len(val_smiles)} samples")
print(f"Test data: {len(test_smiles)} samples")
print(f"Number of tasks: {y_train.shape[1]}")

c:\Users\vinic\anaconda3\envs\graph\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Training data: 14187 samples
Validation data: 1776 samples
Test data: 1776 samples
Number of tasks: 9


##### Building molecular fingerprints

In [4]:
from cv_baselines import smiles_to_ecfp

X_train, invalid_train = smiles_to_ecfp(train_smiles, fp_size=fp_size, radius=radius)
X_val, invalid_val = smiles_to_ecfp(val_smiles, fp_size=fp_size, radius=radius)
X_test, invalid_test = smiles_to_ecfp(test_smiles, fp_size=fp_size, radius=radius)

print('ECFP matrices generated:')
print('X_train:', X_train.shape, '| invalid SMILES:', len(invalid_train))
print('X_val  :', X_val.shape, '| invalid SMILES:', len(invalid_val))
print('X_test :', X_test.shape, '| invalid SMILES:', len(invalid_test))

ECFP matrices generated:
X_train: (14187, 2048) | invalid SMILES: 0
X_val  : (1776, 2048) | invalid SMILES: 0
X_test : (1776, 2048) | invalid SMILES: 0


##### Run selected model with 5-fold CV

In [5]:
from utils import set_seed
from cv_baselines import infer_task_metadata_from_train, fit_predict_multitask_cv

set_seed(seed)
task_type, mc_label_values = infer_task_metadata_from_train(y_train)
print('Task types (0=reg, 1=bin, 2=multiclass):', task_type.tolist())

out = fit_predict_multitask_cv(
    method=method,
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    X_test=X_test,
    y_test=y_test,
    task_type=task_type,
    mc_label_values=mc_label_values,
    model_out_dir=model_out_dir,
    params_out_dir=params_out_dir,
    n_splits=n_splits,
    seed=seed,
    c_value=c_value,
)

trained = sum(1 for t in out['metadata']['tasks'] if t.get('status') == 'trained')
skipped = sum(1 for t in out['metadata']['tasks'] if t.get('status') != 'trained')
print(f"[{method.upper()}] trained tasks: {trained} | skipped tasks: {skipped}")

Task types (0=reg, 1=bin, 2=multiclass): [1, 1, 1, 1, 1, 1, 1, 1, 1]
[RF] trained tasks: 9 | skipped tasks: 0


##### Statistical analysis of selected model

In [6]:
from cv_baselines import evaluate_ml_predictions_cv

# Set calibration=True/False for binary threshold tuning
calibration = False

evaluate_ml_predictions_cv(
    method_name=method,
    task_type=task_type,
    y_true_trainval=out['y_trainval'],
    y_true_test=y_test,
    y_pred_train=out['pred_train'],
    y_pred_val=out['pred_val'],
    y_pred_test=out['pred_test'],
    y_prob_train_mc=out['mc_probs_train'],
    y_prob_val_mc=out['mc_probs_val'],
    y_prob_test_mc=out['mc_probs_test'],
    mc_label_values=mc_label_values,
    calibration=calibration,
)


### RF (5-fold CV train+val, hold-out test)


Task  | Set   | Threshold | Accuracy | Recall | Specificity | PPV  | NPV  | F1   | G-mean | MCC  | PRAUC | AUC
------|-------|-----------|----------|--------|-------------|------|------|------|--------|------|-------|-----
Task 1 | Training   | 0.5000    | 0.7330   | 0.0000 | 1.0000      | 0.0000 | 0.7330 | 0.0000 | 0.0000 | 0.0000 | 0.7613 | 0.9136
Task 1 | Validation | 0.5000    | 0.7330   | 0.0000 | 1.0000      | 0.0000 | 0.7330 | 0.0000 | 0.0000 | 0.0000 | 0.6788 | 0.8792
Task 1 | Test       | 0.5000    | 0.7222   | 0.0000 | 1.0000      | 0.0000 | 0.7222 | 0.0000 | 0.0000 | 0.0000 | 0.7706 | 0.8865
Task 2 | Training   | 0.5000    | 0.6713   | 0.0000 | 1.0000      | 0.0000 | 0.6713 | 0.0000 | 0.0000 | 0.0000 | 0.6936 | 0.8322
Task 2 | Validation | 0.5000    | 0.6713   | 0.0000 | 1.0000      | 0.0000 | 0.6713 | 0.0000 | 0.0000 | 0.0000 | 0.6024 | 0.7908
Task 2 | Test       | 0.5000    | 0.6600   | 0.0000 | 1.0000      | 0.0000 | 0.6600 | 0.0000 | 0.0000 | 0.0000 | 0.7439 | 0.8743
Task 3 | Training   | 0.5000    | 0.5932   | 0.0852 | 0.9777      | 0.7429 | 0.5854 | 0.1529 | 0.2887 | 0.1437 | 0.6385 | 0.7269
Task 3 | Validation | 0.5000    | 0.5946   | 0.0918 | 0.9752      | 0.7368 | 0.5866 | 0.1633 | 0.2992 | 0.1472 | 0.5882 | 0.6784
Task 3 | Test       | 0.5000    | 0.6364   | 0.0667 | 1.0000      | 1.0000 | 0.6267 | 0.1250 | 0.2582 | 0.2044 | 0.5980 | 0.6734
Task 4 | Training   | 0.5000    | 0.6083   | 0.0000 | 1.0000      | 0.0000 | 0.6083 | 0.0000 | 0.0000 | 0.0000 | 0.6114 | 0.6727
Task 4 | Validation | 0.5000    | 0.6083   | 0.0000 | 1.0000      | 0.0000 | 0.6083 | 0.0000 | 0.0000 | 0.0000 | 0.6011 | 0.6659
Task 4 | Test       | 0.5000    | 0.6175   | 0.0000 | 1.0000      | 0.0000 | 0.6175 | 0.0000 | 0.0000 | 0.0000 | 0.5853 | 0.6509
Task 5 | Training   | 0.5000    | 0.7277   | 0.7740 | 0.6678      | 0.7506 | 0.6958 | 0.7621 | 0.7189 | 0.4441 | 0.8709 | 0.8201
Task 5 | Validation | 0.5000    | 0.7204   | 0.7675 | 0.6594      | 0.7443 | 0.6871 | 0.7558 | 0.7114 | 0.4292 | 0.8571 | 0.8058
Task 5 | Test       | 0.5000    | 0.7071   | 0.7436 | 0.6613      | 0.7342 | 0.6721 | 0.7389 | 0.7012 | 0.4056 | 0.8661 | 0.8112
Task 6 | Training   | 0.5000    | 0.7524   | 0.0000 | 1.0000      | 0.0000 | 0.7524 | 0.0000 | 0.0000 | 0.0000 | 0.4476 | 0.6976
Task 6 | Validation | 0.5000    | 0.7524   | 0.0000 | 1.0000      | 0.0000 | 0.7524 | 0.0000 | 0.0000 | 0.0000 | 0.4199 | 0.6705
Task 6 | Test       | 0.5000    | 0.7276   | 0.0000 | 1.0000      | 0.0000 | 0.7276 | 0.0000 | 0.0000 | 0.0000 | 0.4951 | 0.7195
Task 7 | Training   | 0.5000    | 0.6954   | 0.0000 | 1.0000      | 0.0000 | 0.6954 | 0.0000 | 0.0000 | 0.0000 | 0.7293 | 0.8940
Task 7 | Validation | 0.5000    | 0.6954   | 0.0000 | 1.0000      | 0.0000 | 0.6954 | 0.0000 | 0.0000 | 0.0000 | 0.6939 | 0.8562
Task 7 | Test       | 0.5000    | 0.6000   | 0.0000 | 1.0000      | 0.0000 | 0.6000 | 0.0000 | 0.0000 | 0.0000 | 0.8245 | 0.8611
Task 8 | Training   | 0.5000    | 0.7044   | 0.0000 | 1.0000      | 0.0000 | 0.7044 | 0.0000 | 0.0000 | 0.0000 | 0.6346 | 0.7744
Task 8 | Validation | 0.5000    | 0.7044   | 0.0000 | 1.0000      | 0.0000 | 0.7044 | 0.0000 | 0.0000 | 0.0000 | 0.5442 | 0.7164
Task 8 | Test       | 0.5000    | 0.6667   | 0.0000 | 1.0000      | 0.0000 | 0.6667 | 0.0000 | 0.0000 | 0.0000 | 0.7958 | 0.8730
Task 9 | Training   | 0.5000    | 0.6488   | 0.0000 | 1.0000      | 0.0000 | 0.6488 | 0.0000 | 0.0000 | 0.0000 | 0.5794 | 0.6763
Task 9 | Validation | 0.5000    | 0.6488   | 0.0000 | 1.0000      | 0.0000 | 0.6488 | 0.0000 | 0.0000 | 0.0000 | 0.5350 | 0.6458
Task 9 | Test       | 0.5000    | 0.6854   | 0.0000 | 1.0000      | 0.0000 | 0.6854 | 0.0000 | 0.0000 | 0.0000 | 0.5327 | 0.6412
Global | Training   | 0.5000    | 0.6477   | 0.0858 | 0.9830      | 0.7503 | 0.6431 | 0.1539 | 0.2903 | 0.1644 | 0.5854 | 0.6815
Global | Validation | 0.5000    | 0.6472   | 0.0853 | 0.9825      | 0.7440 | 0.6429 | 0.1531 | 0.2896 | 0.1620 | 0.5750 | 0.6732
Global | Test       | 0.5000    | 0.6537   | 0.0770 | 0.9845      | 0.7407 | 0.6503 | 0.1395 | 0.2754 | 0.1551 | 0.5597 | 0.6631


No multiclass tasks detected.
No regression tasks detected.
